In [14]:
%pip install requests beautifulsoup4 pymysql

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [29]:
import csv, time
from pathlib import Path
from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup

BASE_URL = "http://www.bspider.top/anjuke/"
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/155.0.0.0 Safari/537.36"),
    "Referer": "http://www.bspider.top/",   
}
FIELDS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]
CSV_PATH = Path("data") / "houseinfo.csv"

def fetch_soup(url):
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    r.encoding = r.apparent_encoding
    return BeautifulSoup(r.text, "html.parser"), r

soup, r = fetch_soup(BASE_URL)
print(r.status_code, r.encoding, soup.title.get_text(strip=True))
print("div.property ", len(soup.select("div.property")))


200 utf-8 重庆二手房房产网，重庆二手房出售，重庆买房购房交易信息 - 58安居客
div.property  60


In [ ]:
#__layout > div > section > section.list-main > section.list-left > section:nth-child(4) > div:nth-child(1)
#__layout > div > section > section.list-main > section.list-left > section:nth-child(4) > div:nth-child(1) > a > div.property-content > div.property-content-detail > div.property-content-title > h3

In [30]:
ROW = ("#__layout > div > section > section.list-main > section.list-left > "
       "section:nth-child(4) > div:nth-child(1)")
DETAIL = " > a > div.property-content > div.property-content-detail"
INFO = DETAIL + " > section > div.property-content-info"

COPIED = {
    "title":    ROW + DETAIL + " > div.property-content-title > h3",
    "house":    ROW + INFO + ".property-content-info-comm > p.property-content-info-comm-name",
    "address":  ROW + INFO + ".property-content-info-comm > p.property-content-info-comm-address",
    "struct":   ROW + INFO + " > p.property-content-info-attribute",
    "area":     ROW + INFO + " > p.property-content-info-text:nth-of-type(2)",
    "floor":    ROW + INFO + " > p.property-content-info-text:nth-of-type(4)",
    "maketime": ROW + INFO + " > p.property-content-info-text:nth-of-type(5)",
    "price":    ROW + " > a > div.property-content > div.property-price > p.property-price-average",
}
for field, sel in COPIED.items():
    nodes = soup.select(sel)
    print(f"{field:9s} {len(nodes)}  {nodes[0].get_text(strip=True)}")


title     1  万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
house     1  万科金域学府翰林菁华里
address   1  南岸学府大道二塘路59号
struct    1  3室2厅1卫
area      1  89.9㎡
floor     1  低层(共27层)
maketime  1  2019年建造
price     1  16465元/㎡


In [36]:
rows = soup.select("div.property")         
print("本页房源数", len(rows))

def parse_row(row):
    def text(sel):
        node = row.select_one(sel)
        return node.get_text(strip=True) if node else ""

    infos = [p.get_text(strip=True) for p in row.select(".property-content-info-text")]
    area = floor = maketime = ""
    for item in infos:
        if "㎡" in item or "平米" in item:
            area = item                    
        elif "层" in item:
            floor = item                    
        elif "建造" in item:
            maketime = item               

    return {
        "title":    text("h3"),
        "house":    text(".property-content-info-comm-name"),
        "address":  text(".property-content-info-comm-address"),  
        "struct":   text(".property-content-info-attribute"),
        "area":     area,
        "floor":    floor,
        "maketime": maketime,
        "price":    text(".property-price-average"),          
    }

first = parse_row(rows[0])
for f in FIELDS:
    print(f"{f:9s} {first[f]}")
print("属性组原文：", [p.get_text(strip=True) for p in rows[0].select(".property-content-info-text")])


本页房源数 60
title     万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
house     万科金域学府翰林菁华里
address   南岸学府大道二塘路59号
struct    3室2厅1卫
area      89.9㎡
floor     低层(共27层)
maketime  2019年建造
price     16465元/㎡
属性组原文： ['3室2厅1卫', '89.9㎡', '南北', '低层(共27层)', '2019年建造']


In [18]:
records, url, page = [], BASE_URL, 1
visited, MAX_PAGES = {BASE_URL}, 20

while url and page <= MAX_PAGES:
    if page > 1:
        soup, r = fetch_soup(url)
        rows = soup.select("div.property")
    if not rows:
        print("！无房源节点", soup.title.get_text(strip=True)); break

    records.extend(parse_row(row) for row in rows)
    nxt = soup.select_one("div.pagination a.next")
    href = nxt.get("href") if nxt else None
    next_url = urljoin(url, href) if href else None
    print(f"第{page}页 本页{len(rows)}条 累计{len(records)}条 href={href!r} next_url={next_url}")

    if nxt is None:
        print("！分页结构异常，中止"); break
    if "next-active" not in (nxt.get("class") or []):
        print("末页：下一页按钮已禁用，采集正常结束"); break
    if not next_url.startswith(BASE_URL) or next_url in visited:
        print("！越站或重复，中止"); break
    visited.add(next_url); url = next_url; page += 1
    time.sleep(1)

seen, uniq = set(), []
for rec in records:
    key = tuple(rec[f] for f in FIELDS)
    if key not in seen:
        seen.add(key); uniq.append(rec)
print("去重前", len(records), "去重后", len(uniq), "重复", len(records) - len(uniq))

CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(CSV_PATH, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f)
    w.writerow(FIELDS)
    for rec in uniq:
        w.writerow([rec[f] for f in FIELDS])

with open(CSV_PATH, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    print("表头", reader.fieldnames)
    csv_rows = list(reader)
print("CSV 数据行数", len(csv_rows))
for label, idx in [("第1页第1条", 0), ("第2页第1条", 60), ("最后一条", len(csv_rows)-1)]:
    print(label, " | ".join(csv_rows[idx][f] for f in FIELDS))


第1页 本页60条 累计60条 href='?page=2' next_url=http://www.bspider.top/anjuke/?page=2
第2页 本页60条 累计120条 href='?page=3' next_url=http://www.bspider.top/anjuke/?page=3
第3页 本页60条 累计180条 href='?page=4' next_url=http://www.bspider.top/anjuke/?page=4
第4页 本页60条 累计240条 href='?page=5' next_url=http://www.bspider.top/anjuke/?page=5
第5页 本页60条 累计300条 href='?page=6' next_url=http://www.bspider.top/anjuke/?page=6
第6页 本页60条 累计360条 href='?page=7' next_url=http://www.bspider.top/anjuke/?page=7
第7页 本页60条 累计420条 href='?page=8' next_url=http://www.bspider.top/anjuke/?page=8
第8页 本页60条 累计480条 href='?page=9' next_url=http://www.bspider.top/anjuke/?page=9
第9页 本页60条 累计540条 href='?page=10' next_url=http://www.bspider.top/anjuke/?page=10
第10页 本页60条 累计600条 href='#' next_url=http://www.bspider.top/anjuke/?page=10
末页：下一页按钮已禁用，采集正常结束
去重前 600 去重后 498 重复 102
表头 ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']
CSV 数据行数 498
第1页第1条 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器 | 万科金域学府翰林菁华里 | 南岸学府大道二塘路59号 | 3室2厅1卫 |

In [19]:
import os
print("内核当前工作目录:", os.getcwd())
os.chdir("/home/pwuser/workspace/bigdata-collection-labs-112404260124/lab02")
print("已切换到:", os.getcwd())


内核当前工作目录: /home/pwuser/workspace/bigdata-collection-labs-112404260124/lab02
已切换到: /home/pwuser/workspace/bigdata-collection-labs-112404260124/lab02


In [28]:
seen, uniq = set(), []
for rec in records:
    key = tuple(rec[f] for f in FIELDS)  
    if key not in seen:
        seen.add(key)
        uniq.append(rec)
print("去重前", len(records), "去重后", len(uniq), "重复", len(records) - len(uniq))

CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(CSV_PATH, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(FIELDS)                 
    for rec in uniq:
        writer.writerow([rec[f] for f in FIELDS])
print("已写入", CSV_PATH)

with open(CSV_PATH, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    print("表头", reader.fieldnames)
    csv_rows = list(reader)                
print("CSV 数据行数", len(csv_rows))
for label, idx in [("第1页第1条", 0), ("第2页第1条", 60), ("最后一条", len(csv_rows) - 1)]:
    print(label, "|", " | ".join(csv_rows[idx][f] for f in FIELDS))
bad = [i for i, row in enumerate(csv_rows) if not all(row[f] for f in FIELDS)]
print("空值行:", bad if bad else "无")

去重前 600 去重后 498 重复 102
已写入 data/houseinfo.csv
表头 ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']
CSV 数据行数 498
第1页第1条 | 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器 | 万科金域学府翰林菁华里 | 南岸学府大道二塘路59号 | 3室2厅1卫 | 89.9㎡ | 低层(共27层) | 2019年建造 | 16465元/㎡
第2页第1条 | 万科物业自营 朝中庭 业主诚心出售 | 万科金域学府翰林菁华里 | 南岸学府大道二塘路59号 | 3室2厅1卫 | 85.7㎡ | 低层(共27层) | 2018年建造 | 16914元/㎡
最后一条 | 恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售 | 恒大照母山山水城 | 渝北照母山恒康路9号 | 3室2厅1卫 | 90.2㎡ | 低层(共32层) | 2015年建造 | 15530元/㎡
空值行: 无


In [21]:
# 3-1 从磁盘重新读取 CSV，检查表头
with open(CSV_PATH, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    if reader.fieldnames != FIELDS:
        raise ValueError(f"表头与 8 个字段不一致：{reader.fieldnames}")
    db_rows = [{k: (row[k] or "").strip() for k in FIELDS} for row in reader]
print("表头检查通过，待入库记录:", len(db_rows))

表头检查通过，待入库记录: 498


In [32]:
# 3-2 pymysql.connect 建立连接，先查一条确认能通
import pymysql

DB_CONFIG = dict(
    host="mysql",           # 课程数据库服务名（容器网络内解析）
    port=3306,
    database="smartpark_lab",
    user="student",
    password="123456",
    charset="utf8mb4",
    autocommit=False,       # 事务由程序 commit / rollback 控制
)

conn = pymysql.connect(**DB_CONFIG)
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM houseinfo")
print("连接正常，入库前条数:", cur.fetchone()[0])
cur.close()
conn.close()

连接正常，入库前条数: 0


In [23]:
# 3-3 入库函数：比对已有记录 -> executemany -> commit/rollback -> 关闭
INSERT_SQL = ("INSERT INTO houseinfo "
              "(title, house, address, struct, area, floor, maketime, price) "
              "VALUES (%s, %s, %s, %s, %s, %s, %s, %s)")


def load_to_mysql(rows, config=None):
    cfg = DB_CONFIG if config is None else config
    conn = pymysql.connect(**cfg)
    cur = None
    try:
        cur = conn.cursor()
        cur.execute("SELECT title, house, address, struct, area, floor, maketime, price FROM houseinfo")
        existing = {tuple(r) for r in cur.fetchall()}
        new_rows = [r for r in rows if tuple(r[f] for f in FIELDS) not in existing]
        print(f"库内已有 {len(existing)} 条，待插入 {len(new_rows)} 条，跳过 {len(rows) - len(new_rows)} 条")

        if new_rows:
            affected = cur.executemany(INSERT_SQL, [[r[f] for f in FIELDS] for r in new_rows])
            conn.commit()
            print("executemany 影响行数:", affected, "，已 commit")

        cur.execute("SELECT COUNT(*) FROM houseinfo")
        print("入库后条数:", cur.fetchone()[0])
    except Exception as exc:
        conn.rollback()
        print("写入失败，已 rollback：", exc)
        raise
    finally:
        if cur is not None:
            cur.close()
        conn.close()

In [33]:
# 3-4 试插 3 条
load_to_mysql(db_rows[:3])

# 试插后立刻查样本核对
conn = pymysql.connect(**DB_CONFIG)
cur = conn.cursor()
cur.execute("SELECT id, title, area, price FROM houseinfo ORDER BY id LIMIT 5")
for row in cur.fetchall():
    print(row)
cur.close()
conn.close()

库内已有 0 条，待插入 3 条，跳过 0 条
executemany 影响行数: 3 ，已 commit
入库后条数: 3
(1, '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', '89.9㎡', '16465元/㎡')
(2, '黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住', '94.5㎡', '14815元/㎡')
(3, '鸳鸯16年次新小区 大雅云居山刚需经典3房户型 随时看房诚售', '88㎡', '16487元/㎡')


In [34]:
# 3-5 全量写入
load_to_mysql(db_rows)

库内已有 3 条，待插入 495 条，跳过 3 条
executemany 影响行数: 495 ，已 commit
入库后条数: 498


In [35]:
# 3-6 重跑同一批数据（防重复验证）
load_to_mysql(db_rows)

库内已有 498 条，待插入 0 条，跳过 498 条
入库后条数: 498


```sql
USE smartpark_lab;
SELECT COUNT(*) AS after_count FROM houseinfo;   -- 预期 498
SELECT id, title, house, address, struct, area, floor, maketime, price
FROM houseinfo ORDER BY id LIMIT 5;
```

| 项目 | 预期 |
|---|---|
| 入库前条数 | 0 |
| 试插 | 3 条 |
| 全量写入 | 495 条 |
| 入库后 after_count | 498（= CSV 数据行数） |
| 重跑同一批 | 待插入 0 条，条数不变 |